## EDA Statsbomb

Ziel dieses Notebooks ist es, zwei Kennzahlen für das Statistik-Tab des GSA zu entwickeln, die auf die Quelle **Statsbomb** zugreifen und zur aktuellen Architektur des Tabs passen. 

Hinweis: Dieses Notebook wird mit Gemini und Claude entwickelt.

----------------------------------

Bisherige Architektur (siehe Notebook `EDA_Openliga_DB_minimal.ipynb`)
```

┌─────────────────────────────────────┐
│  Filter: [Saison ▾]  [Verein ▾]      │
└─────────────────────────────────────┘

┌──────────┬──────────┬──────────┐
│ Platz    │ Punkte   │ Siege    │
│   3      │   68     │   21     │
└──────────┴──────────┴──────────┘

┌──────────┬──────────┐
│ Tore     │ Gegentore│
│   89     │   32     │
└──────────┴──────────┘

[Barchart: Tore vs. Gegentore über alle Vereine]



--------------------------------

```
┌─────────────────────────────────────┐
│  Filter: [Saison ▾]  [Verein ▾]      │
└─────────────────────────────────────┘

┌──────────┬──────────┬──────────┐
│ Platz    │ Punkte   │ Siege    │
│   3      │   68     │   21     │
└──────────┴──────────┴──────────┘

┌─────────────────────┬─────────────────────┐
│ Tore (OpenLigaDB)   │ Gegentore           │
│   89 (xG: 82.4)     │   32                │
├─────────────────────┼─────────────────────┤
│ Chancen-Verwertung  │ Druck-Resistenz     │
│   +6.6 (Top-Wert)   │   76.4 % (StatsBomb)│
└─────────────────────┴─────────────────────┘

[Barchart: Tore vs. Gegentore über alle Vereine]

### Erklärung der neuen Kennzahlen (xG-Effizienz und Druck-Resistenz)

Die Angaben "Tore" und "Gegentore" (OpenLigaDB) werden durch "xG-Effizienz" und "Druck-Resistenz" ergänzt (StatsBomb-Daten). Diese bieten eine taktische Ursachenanalyse über eine gesamte Bundesliga-Saison (34 Spieltage) hinweg.

---

#### 1. Chancen-Qualität (Expected Goals - xG) & Tor-Effizienz

Der mathematische Wert **Expected Goals (xG)** misst die historische Wahrscheinlichkeit, mit der ein Schuss aus einer bestimmten Position (Winkel, Distanz, Gegnerdruck) im Tor landet. Ein Wert von `0.1` bedeutet beispielsweise, dass dieser Schuss statistisch gesehen in 10 % der Fälle zu einem Tor führt.

Für die Saison-Ebene nutzen wir die **Tor-Effizienz** ($\Delta\text{xG}$):
$$\text{Tor-Effizienz} = \text{Tatsächlich erzielte Tore} - \text{Erwartbare Tore (Gesamt-xG)}$$

###### Interpretation für die Saison-Analyse:
Da über 34 Spieltage hinweg kurzfristiges Glück mathematisch weitgehend herausgefiltert wird, zeigt diese Kennzahl die echte Qualität der Offensive:

* **Positiver Wert (z. B. $+6.6$):** *Extrem effizient.* Das Team kann schwierige Chancen verwerten oder behält in Schlüsselmomenten sehr gut die Nerven.
* **Wert um Null (z. B. $\pm0.0$):** *Präzise im Soll.* Das Team erzielt exakt so viele Tore, wie es sich laut Qualität der Chancen statistisch erarbeitet hat.
* **Negativer Wert (z. B. $-5.1$):** *Chancenwucher / Abschluss-Schwäche.* Das Team erarbeitet sich zwar hochkarätige Chancen, vergibt diese aber unterdurchschnittlich oft.

---

#### 2. Druck-Resistenz (Pressure Resistance)

StatsBomb erfasst jede Aktion, in der ein Spieler vom Gegner unter direkten physischen oder taktischen Druck gesetzt wird (*Pressure Event*). Die bloße Anzahl dieser Situationen ist über eine Saison hinweg zu hoch, um aussagekräftig zu sein.

Daher die Kennzahl **Druck-Resistenz (%)**:
$$\text{Druck-Resistenz} = \left( \frac{\text{Erfolgreiche Folge-Aktionen unter Druck}}{\text{Gegnerische Drucksituationen gesamt}} \right) \times 100$$

###### Interpretation für die Saison-Analyse:
Diese Metrik liefert die taktische Erklärung, die direkt unter der **Gegentore-Kachel** platziert wird:

- **Hohe Druck-Resistenz (z. B. $>75\%$):** Das Team kann sich auch in engen Räumen spielerisch befreien. Es verliert im Spielaufbau selten den Ball in der eigenen Hälfte, was die Anzahl der gefährlichen gegnerischen Konter und somit die Gegentore drastisch minimiert.
- **Niedrige Druck-Resistenz:** Das Team gerät bei gegnerischem Pressing in Panik, produziert Fehlpässe im defensiven Drittel und lädt den Gegner zu einfachen Torchancen ein.

---

Zusammenhang zu den beiden anderen Tabs:
1. **Verbindung zu Tab 2 (Clustering):** Teams mit ähnlicher Druck-Resistenz landen oft im selben spielerischen Cluster (z. B. "Ballbesitz-dominierte Teams" vs. "Konter-Teams").
2. Gewonnene Kennzahl kann auch für die Integration in das Chattab interessant sein.

------------------------------------

## Umsetzung der beiden neuen Kennzahlen fürs Statistik-Tab

Vorgehensweise: Aus den verschachtelten json-Daten von Statsbomb zwei "Typen" raussuchen:
- **Schüsse (Shot)**: Jedes Mal, wenn geschossen wird, gibt StatsBomb die erzielten goals (0 oder 1) und den eingebauten statsbomb_xg-Wert (eine Zahl zwischen 0 und 1).
- **Drucksituationen (Pressure)**: Jedes Mal, wenn ein Spieler bedrängt wird, herausfinden, ob das direkt darauffolgende Event desselben Teams erfolgreich war (kein Fehlpass, kein Ballverlust).

### Setup

In [10]:
import pandas as pd
from statsbombpy import sb

### 1. Schritt: Oberste Ebene der verschachtelten Datenstruktur: Wettbewerbe

- Leitfrage: Welche Daten zur 1. Bundesliga sind verfügbar? Welche Saisonen kann man kostenlos abrufen?

In [5]:
# Alle Wettbewerbe abrufen
all_comps = sb.competitions()

# Nur nach der deutschen 1. Bundesliga filtern
bundesliga_seasons = all_comps[all_comps['competition_name'] == '1. Bundesliga']

print("Verfügbare Bundesliga-Saisons bei StatsBomb:")
display(bundesliga_seasons[['competition_id', 'season_id', 'season_name']])

Verfügbare Bundesliga-Saisons bei StatsBomb:


c:\Users\Annette\00_Data_Science\06_NLP_GenAI\05_sandbox_task_2\.venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_id,season_id,season_name
0,9,281,2023/2024
1,9,27,2015/2016


Frei zugänglich gibt es nur Daten für die Saison 2015/16 (nicht relevant) und für 2023/24 (relevant, da im Filter des Statistik-Tab enthalten). Wählt der Nutzer über die Filter eine andere Saison, muss ein Strich für die Angaben "Chancen-Verwertung" und "Druck-Resistenz" erscheinen. 

----------------------------

### 2. Schritt: Spiele zur Saison 23/24 rausfiltern

Um dem User im Dashboard eine Auswahlliste (Dropdown) mit den richtigen Vereinen anzubieten, müssen zuerst alle Spiele der ausgewählten Saison geholt werden und daraus die Teamnamen herausgefiltert werden.

In [6]:
# 1. Wir holen uns alle Spiele für die Bundesliga (9) in der Saison 23/24 (281)
matches_23_24 = sb.matches(competition_id=9, season_id=281)

# 2. Sammeln aller Vereine, die entweder ein Heimspiel oder ein Auswärtsspiel hatten
heim_teams = matches_23_24['home_team'].unique()
auswaerts_teams = matches_23_24['away_team'].unique()

# 3. Beide Listen zusammenführen, Duplikate löschen (mit set) und alphabetisch sortieren
alle_vereine = sorted(list(set(heim_teams) | set(auswaerts_teams)))

print(f"Gefundene Vereine für die Saison 23/24 ({len(alle_vereine)} Teams):")
print(alle_vereine)

c:\Users\Annette\00_Data_Science\06_NLP_GenAI\05_sandbox_task_2\.venv\Lib\site-packages\statsbombpy\api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


Gefundene Vereine für die Saison 23/24 (18 Teams):
['Augsburg', 'Bayer Leverkusen', 'Bayern Munich', 'Bochum', 'Borussia Dortmund', 'Borussia Mönchengladbach', 'Darmstadt 98', 'Eintracht Frankfurt', 'FC Heidenheim', 'FC Köln', 'FSV Mainz 05', 'Freiburg', 'Hoffenheim', 'RB Leipzig', 'Union Berlin', 'VfB Stuttgart', 'Werder Bremen', 'Wolfsburg']


Schreibweise der Vereine beachten bzw. anpassen, damit der Filter später funktioniert! 

- Beispiel: Bayern Munich soll Bayern München heißen!
- Abgleich mit den Daten aus OpenligaDB wichtig!
- Lösung: Übersetzer-Dictionary

In [7]:
# Das Vereins-Mapping: "OpenLigaDB-Name": "StatsBomb-Name"
TEAM_NAME_MAPPING = {
    "FC Bayern München": "Bayern Munich",
    "Bayer 04 Leverkusen": "Bayer Leverkusen",
    "Borussia Dortmund": "Borussia Dortmund",
    "VfB Stuttgart": "VfB Stuttgart",
    "RB Leipzig": "RB Leipzig",
    "Eintracht Frankfurt": "Eintracht Frankfurt",
    "TSG 1899 Hoffenheim": "TSG Hoffenheim",
    "1. FC Heidenheim": "Heidenheim",
    "SV Werder Bremen": "Werder Bremen",
    "SC Freiburg": "Freiburg",
    "FC Augsburg": "Augsburg",
    "VfL Wolfsburg": "Wolfsburg",
    "1. FSV Mainz 05": "Mainz",
    "Borussia Mönchengladbach": "Borussia Mönchengladbach",
    "1. FC Union Berlin": "Union Berlin",
    "VfL Bochum": "Bochum",
    "1. FC Köln": "FC Cologne",
    "SV Darmstadt 98": "Darmstadt"
}

---------------------------------

### KPI Chancen-Verwertung bauen
- inhaltlicher Zusammenhang zu Toren beachten!

-------------------------------------------

### KPI Druck-Resistenz bauen
- inhaltlicher Zusammenhang zu Gegentoren beachten!